# Análise de acessos — Sentinel Access

Exporte `logs_acesso.csv` pelo painel web e coloque o arquivo na pasta raiz do projeto antes de executar as células.

In [ ]:
from pathlib import Path
import pandas as pd

arquivo = Path('..') / 'logs_acesso.csv'
df = pd.read_csv(arquivo, sep=';', encoding='utf-8-sig')
df['ocorrido_em'] = pd.to_datetime(df['ocorrido_em'], errors='coerce', utc=True).dt.tz_convert(None)
df['dia'] = df['ocorrido_em'].dt.date
df = df.sort_values('ocorrido_em')
df.head()

## Entradas e saídas por dia

In [ ]:
entradas_por_dia = df[df['tipo'].eq('ENTRADA')].groupby('dia').size().rename('entradas')
saidas_por_dia = df[df['tipo'].eq('SAIDA')].groupby('dia').size().rename('saidas')
pd.concat([entradas_por_dia, saidas_por_dia], axis=1).fillna(0).astype(int)

## Tentativas negadas e tags desconhecidas

In [ ]:
negados = df[df['tipo'].eq('ACESSO_NEGADO')].groupby('dia').size().rename('acessos_negados')
desconhecidas = df[df['tipo'].eq('TAG_DESCONHECIDA')].groupby('dia').size().rename('tags_desconhecidas')
pd.concat([negados, desconhecidas], axis=1).fillna(0).astype(int)

## Colaboradores com mais tentativas negadas

In [ ]:
df[df['tipo'].eq('ACESSO_NEGADO')].groupby('nome_snapshot').size().sort_values(ascending=False).head(10)

## Permanência estimada por colaborador e dia

O cálculo pareia cada entrada com a saída seguinte do mesmo colaborador. Se faltar uma saída, o intervalo fica aberto e não é somado ao total.

In [ ]:
linhas = []
movimentos = df[df['tipo'].isin(['ENTRADA', 'SAIDA'])]
for (matricula, nome, dia), grupo in movimentos.groupby(['matricula_snapshot', 'nome_snapshot', 'dia'], dropna=False):
    entrada = None
    total = pd.Timedelta(0)
    for _, evento in grupo.sort_values('ocorrido_em').iterrows():
        if evento['tipo'] == 'ENTRADA':
            entrada = evento['ocorrido_em']
        elif evento['tipo'] == 'SAIDA' and entrada is not None:
            if evento['ocorrido_em'] >= entrada:
                total += evento['ocorrido_em'] - entrada
            entrada = None
    linhas.append({'dia': dia, 'matricula': matricula, 'colaborador': nome,
                   'permanencia_minutos': round(total.total_seconds() / 60, 2),
                   'intervalo_aberto': entrada is not None})
pd.DataFrame(linhas)